In [1]:
import numpy as np 
import pandas as pd

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AffinityPropagation
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
import warnings
warnings.filterwarnings("ignore")
import plotly as py
import plotly.graph_objs as go
import os
py.offline.init_notebook_mode(connected = True)
#print(os.listdir("../input"))
import datetime as dt
import missingno as msno
plt.rcParams['figure.dpi'] = 140

In [3]:

file_path = r"C:\Users\jayar\Downloads\project\Netflix-Content-Strat\Week-3-task\netflix_titles (1).csv"

df = pd.read_csv(file_path)
df.head(3)


,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...


In [4]:
print(f"Number of rows: {len(df)}")

Number of rows: 8807


In [5]:
# count missing values in each column
null_counts = df.isnull().sum()
print(null_counts)

show_id            0
type               0
title              0
director        2634
cast             825
country          831
date_added        10
release_year       0
rating             4
duration           3
listed_in          0
description        0
dtype: int64


In [6]:
# Explicit null handling per request: fill specific categorical columns,
# drop remaining low-count nulls, and convert date fields to datetime.
df['director'] = df['director'].fillna('Unknown')
df['cast'] = df['cast'].fillna('Not available')
df['country'] = df['country'].fillna('Not available')
df['rating'] = df['rating'].fillna('Not rated')

# Drop rows with nulls in columns that have low null counts
# (as requested: 'rating', 'duration', 'date_added')
df = df.dropna(subset=['rating', 'duration', 'date_added'])
print(f"Dataset shape after dropping nulls in rating/duration/date_added: {df.shape}")
df

Dataset shape after dropping nulls in rating/duration/date_added: (8794, 12)


,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,Not available,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,Unknown,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",Not available,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
3,s4,TV Show,Jailbirds New Orleans,Unknown,Not available,Not available,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo..."
4,s5,TV Show,Kota Factory,Unknown,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...
...,...,...,...,...,...,...,...,...,...,...,...,...
8802,s8803,Movie,Zodiac,David Fincher,"Mark Ruffalo, Jake Gyllenhaal, Robert Downey J...",United States,"November 20, 2019",2007,R,158 min,"Cult Movies, Dramas, Thrillers","A political cartoonist, a crime reporter and a..."
8803,s8804,TV Show,Zombie Dumb,Unknown,Not available,Not available,"July 1, 2019",2018,TV-Y7,2 Seasons,"Kids' TV, Korean TV Shows, TV Comedies","While living alone in a spooky town, a young g..."
8804,s8805,Movie,Zombieland,Ruben Fleischer,"Jesse Eisenberg, Woody Harrelson, Emma Stone, ...",United States,"November 1, 2019",2009,R,88 min,"Comedies, Horror Movies",Looking to survive in a world taken over by zo...
8805,s8806,Movie,Zoom,Peter Hewitt,"Tim Allen, Courteney Cox, Chevy Chase, Kate Ma...",United States,"January 11, 2020",2006,PG,88 min,"Children & Family Movies, Comedies","Dragged from civilian life, a former superhero..."


In [7]:
# 1. parse date_added normally
# 2. parse release_year using a year-only format; avoid the integer-nanoseconds issue

# using format='%Y' ensures values like 2019 become 2019-01-01
# instead of being interpreted as nanoseconds since epoch (which gave 1970 dates)
df['date_added']   = pd.to_datetime(df['date_added'],   errors='coerce')
df['release_year'] = pd.to_datetime(df['release_year'], format='%Y', errors='coerce')

# keep full timestamp for release_year; extract year later if needed

# verify
def _check_dtypes():
    print(df[['date_added','release_year']].dtypes)

_check_dtypes()
#   date_added      datetime64[ns]
#   release_year    datetime64[ns]
#   dtype: object

# sample values to confirm correct parsing
print(df[['release_year']].head())

# and df.info() should now report datetime dtype(s) for the columns

# Drop rows where conversion produced NaT (non-date values)
df = df.dropna(subset=['date_added', 'release_year'])
print(f"Dataset shape after ensuring date formats: {df.shape}")

df

date_added      datetime64[us]
release_year    datetime64[us]
dtype: object
  release_year
0   2020-01-01
1   2021-01-01
2   2021-01-01
3   2021-01-01
4   2021-01-01
Dataset shape after ensuring date formats: (8706, 12)


,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,Not available,United States,2021-09-25,2020-01-01,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,Unknown,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,2021-09-24,2021-01-01,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",Not available,2021-09-24,2021-01-01,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
3,s4,TV Show,Jailbirds New Orleans,Unknown,Not available,Not available,2021-09-24,2021-01-01,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo..."
4,s5,TV Show,Kota Factory,Unknown,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,2021-09-24,2021-01-01,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...
...,...,...,...,...,...,...,...,...,...,...,...,...
8802,s8803,Movie,Zodiac,David Fincher,"Mark Ruffalo, Jake Gyllenhaal, Robert Downey J...",United States,2019-11-20,2007-01-01,R,158 min,"Cult Movies, Dramas, Thrillers","A political cartoonist, a crime reporter and a..."
8803,s8804,TV Show,Zombie Dumb,Unknown,Not available,Not available,2019-07-01,2018-01-01,TV-Y7,2 Seasons,"Kids' TV, Korean TV Shows, TV Comedies","While living alone in a spooky town, a young g..."
8804,s8805,Movie,Zombieland,Ruben Fleischer,"Jesse Eisenberg, Woody Harrelson, Emma Stone, ...",United States,2019-11-01,2009-01-01,R,88 min,"Comedies, Horror Movies",Looking to survive in a world taken over by zo...
8805,s8806,Movie,Zoom,Peter Hewitt,"Tim Allen, Courteney Cox, Chevy Chase, Kate Ma...",United States,2020-01-11,2006-01-01,PG,88 min,"Children & Family Movies, Comedies","Dragged from civilian life, a former superhero..."


In [8]:
# count missing values in each column
null_counts = df.isnull().sum()
print(null_counts)

show_id         0
type            0
title           0
director        0
cast            0
country         0
date_added      0
release_year    0
rating          0
duration        0
listed_in       0
description     0
dtype: int64


In [9]:
# Check for inconsistent categories in the 'type' column
print("Unique values in 'type' column:")
print(df['type'].unique())
print(f"\nValue counts:\n{df['type'].value_counts()}")

# Standardize the 'type' column (fix inconsistencies like 'Tv Show' vs 'TV Show')
df['type'] = df['type'].str.replace('Tv Show', 'TV Show', case=False)
print(f"\nAfter standardization:")
print(df['type'].value_counts())

# Check for other categorical columns with potential inconsistencies
print("\n\nChecking 'rating' column:")
print(f"Unique ratings: {df['rating'].nunique()}")
print(f"Ratings:\n{df['rating'].value_counts()}")
# Check for inconsistencies in other categorical columns
print("\n\nChecking other categorical columns for inconsistencies:")
for col in ['rating', 'country', 'listed_in']:
    if col in df.columns:
        print(f"\n{col.upper()}:")
        print(f"Unique values: {df[col].nunique()}")
        print(df[col].value_counts().head())

# Standardize whitespace issues (leading/trailing spaces) for string/object columns only
str_cols = df.select_dtypes(include=['object', 'string']).columns
for col in str_cols:
    df[col] = df[col].apply(lambda x: x.strip() if isinstance(x, str) else x)
    df[col] = df[col].astype(object)

# Convert any pandas 'string' (extension) dtype to plain Python object to avoid `str` dtype
import pandas as pd
for col in df.columns:
    if pd.api.types.is_string_dtype(df[col].dtype):
        df[col] = df[col].astype(object)

# ------------------------------------------------------------------
# Handle duration: split into numeric value and unit
if 'duration' in df.columns:
    df['duration'] = df['duration'].astype(str)
    df['duration_value'] = df['duration'].str.extract(r"(\d+)")
    df['duration_value'] = pd.to_numeric(df['duration_value'], errors='coerce')
    df['duration_unit'] = df['duration'].str.extract(r"([A-Za-z]+)")

    print("\nDuration column split into value and unit")
    # Convert categorical columns to lowercase for standardization
    cat_cols = ['type', 'rating', 'country', 'listed_in', 'director', 'cast']
    for col in cat_cols:
        if col in df.columns:
            df[col] = df[col].str.lower()
print("\n\nData standardization complete!")
print(f"Final dataset shape: {df.shape}")
print("\nSummary of data processing:")
print(f"✓ Data standardized and whitespace trimmed")
print(f"✓ No rows removed - all data preserved with imputation")

Unique values in 'type' column:
<StringArray>
['Movie', 'TV Show']
Length: 2, dtype: str

Value counts:
type
Movie      6128
TV Show    2578
Name: count, dtype: int64

After standardization:
type
Movie      6128
TV Show    2578
Name: count, dtype: int64


Checking 'rating' column:
Unique ratings: 15
Ratings:
rating
TV-MA        3183
TV-14        2133
TV-PG         838
R             799
PG-13         490
TV-Y7         330
TV-Y          300
PG            287
TV-G          212
NR             78
G              41
TV-Y7-FV        5
Not rated       4
NC-17           3
UR              3
Name: count, dtype: int64


Checking other categorical columns for inconsistencies:

RATING:
Unique values: 15
rating
TV-MA    3183
TV-14    2133
TV-PG     838
R         799
PG-13     490
Name: count, dtype: int64

COUNTRY:
Unique values: 746
country
United States     2775
India              971
Not available      827
United Kingdom     403
Japan              241
Name: count, dtype: int64

LISTED_IN:
Unique va

In [10]:
df[['type','duration','duration_value','duration_unit']].head()

,type,duration,duration_value,duration_unit
0,movie,90 min,90,min
1,tv show,2 Seasons,2,Seasons
2,tv show,1 Season,1,Season
3,tv show,1 Season,1,Season
4,tv show,2 Seasons,2,Seasons


In [11]:
# verify
print(df[['duration_value','duration_unit']].isnull().sum())

duration_value    0
duration_unit     0
dtype: int64


In [12]:
df

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description,duration_value,duration_unit
0,s1,movie,Dick Johnson Is Dead,kirsten johnson,not available,united states,2021-09-25,2020-01-01,pg-13,90 min,documentaries,"As her father nears the end of his life, filmm...",90,min
1,s2,tv show,Blood & Water,unknown,"ama qamata, khosi ngema, gail mabalane, thaban...",south africa,2021-09-24,2021-01-01,tv-ma,2 Seasons,"international tv shows, tv dramas, tv mysteries","After crossing paths at a party, a Cape Town t...",2,Seasons
2,s3,tv show,Ganglands,julien leclercq,"sami bouajila, tracy gotoas, samuel jouy, nabi...",not available,2021-09-24,2021-01-01,tv-ma,1 Season,"crime tv shows, international tv shows, tv act...",To protect his family from a powerful drug lor...,1,Season
3,s4,tv show,Jailbirds New Orleans,unknown,not available,not available,2021-09-24,2021-01-01,tv-ma,1 Season,"docuseries, reality tv","Feuds, flirtations and toilet talk go down amo...",1,Season
4,s5,tv show,Kota Factory,unknown,"mayur more, jitendra kumar, ranjan raj, alam k...",india,2021-09-24,2021-01-01,tv-ma,2 Seasons,"international tv shows, romantic tv shows, tv ...",In a city of coaching centers known to train I...,2,Seasons
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8802,s8803,movie,Zodiac,david fincher,"mark ruffalo, jake gyllenhaal, robert downey j...",united states,2019-11-20,2007-01-01,r,158 min,"cult movies, dramas, thrillers","A political cartoonist, a crime reporter and a...",158,min
8803,s8804,tv show,Zombie Dumb,unknown,not available,not available,2019-07-01,2018-01-01,tv-y7,2 Seasons,"kids' tv, korean tv shows, tv comedies","While living alone in a spooky town, a young g...",2,Seasons
8804,s8805,movie,Zombieland,ruben fleischer,"jesse eisenberg, woody harrelson, emma stone, ...",united states,2019-11-01,2009-01-01,r,88 min,"comedies, horror movies",Looking to survive in a world taken over by zo...,88,min
8805,s8806,movie,Zoom,peter hewitt,"tim allen, courteney cox, chevy chase, kate ma...",united states,2020-01-11,2006-01-01,pg,88 min,"children & family movies, comedies","Dragged from civilian life, a former superhero...",88,min


# Task 1: Analyze Netflix Content Growth Over Time

Analyzing how Netflix content has grown over the years by examining the date_added and release_year

In [13]:
# Extract year from date_added
df['year_added'] = df['date_added'].dt.year

# Analyze content growth by year
content_growth = df.groupby('year_added').size().reset_index(name='count')
print("Netflix Content Added Per Year:")
print(content_growth)
print(f"\nTotal titles analyzed: {len(df)}")
print(f"Year range: {df['year_added'].min()} - {df['year_added'].max()}")

Netflix Content Added Per Year:
    year_added  count
0         2008      2
1         2009      2
2         2010      1
3         2011     13
4         2012      3
5         2013     10
6         2014     23
7         2015     73
8         2016    416
9         2017   1163
10        2018   1625
11        2019   1999
12        2020   1878
13        2021   1498

Total titles analyzed: 8706
Year range: 2008 - 2021


In [ ]:
# Visualize overall content growth
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=content_growth['year_added'], 
    y=content_growth['count'],
    mode='lines+markers',
    name='Titles Added',
    line=dict(color='#E50914', width=3),
    marker=dict(size=8)
))

fig.update_layout(
    title='Netflix Content Growth Over Time',
    xaxis_title='Year Added',
    yaxis_title='Number of Titles',
    hovermode='x unified',
    template='plotly_white',
    height=500
)

fig.show()

Wait expired, Browser is being closed by watchdog.


BrowserFailedError: ('The browser seemed to close immediately after starting.', 'You can set the `logging.Logger` level lower to see more output.', 'You may try installing a known working copy of Chrome by running ', '`$ choreo_get_chrome`.It may be your browser auto-updated and will now work upon restart. The browser we tried to start is located at C:\\Program Files (x86)\\Microsoft\\Edge\\Application\\msedge.exe.')

In [ ]:
# Analyze growth by content type
growth_by_type = df.groupby(['year_added', 'type']).size().reset_index(name='count')

fig = go.Figure()

for content_type in growth_by_type['type'].unique():
    data = growth_by_type[growth_by_type['type'] == content_type]
    fig.add_trace(go.Scatter(
        x=data['year_added'],
        y=data['count'],
        mode='lines+markers',
        name=content_type.title(),
        line=dict(width=2),
        marker=dict(size=6)
    ))

fig.update_layout(
    title='Netflix Content Growth by Type (Movie vs TV Show)',
    xaxis_title='Year Added',
    yaxis_title='Number of Titles',
    hovermode='x unified',
    template='plotly_white',
    height=500
)

fig.show()

print("\nContent Added by Type:")
print(growth_by_type.pivot_table(index='year_added', columns='type', values='count', fill_value=0))

Wait expired, Browser is being closed by watchdog.


Wait expired, Browser is being closed by watchdog.


BrowserFailedError: ('The browser seemed to close immediately after starting.', 'You can set the `logging.Logger` level lower to see more output.', 'You may try installing a known working copy of Chrome by running ', '`$ choreo_get_chrome`.It may be your browser auto-updated and will now work upon restart. The browser we tried to start is located at C:\\Program Files (x86)\\Microsoft\\Edge\\Application\\msedge.exe.')

In [ ]:
# Cumulative growth analysis
content_growth['cumulative'] = content_growth['count'].cumsum()

fig = go.Figure()

fig.add_trace(go.Bar(
    x=content_growth['year_added'],
    y=content_growth['count'],
    name='Titles Added That Year',
    marker=dict(color='#E50914'),
    yaxis='y'
))

fig.add_trace(go.Scatter(
    x=content_growth['year_added'],
    y=content_growth['cumulative'],
    name='Cumulative Total',
    line=dict(color='#221F1F', width=3),
    yaxis='y2',
    mode='lines+markers'
))

fig.update_layout(
    title='Netflix Content Growth: Annual vs Cumulative',
    xaxis_title='Year Added',
    yaxis=dict(title='Titles Added That Year'),
    yaxis2=dict(title='Cumulative Total', overlaying='y', side='right'),
    hovermode='x unified',
    template='plotly_white',
    height=500
)

fig.show()

print("\nCumulative Growth Summary:")


Cumulative Growth Summary:


# Task 2: Visualize Distribution of Genres, Ratings, and Content Type

In [ ]:
# Content Type Distribution
type_distribution = df['type'].value_counts().reset_index()
type_distribution.columns = ['type', 'count']

fig = go.Figure(data=[go.Pie(
    labels=type_distribution['type'].str.title(),
    values=type_distribution['count'],
    marker=dict(colors=['#E50914', '#221F1F'])
)])

fig.update_layout(
    title='Netflix Content Distribution by Type',
    height=500
)

fig.show()

print("Content Type Distribution:")
print(type_distribution)
print(f"\nPercentage:")

Content Type Distribution:
      type  count
0    movie   6128
1  tv show   2578

Percentage:


In [ ]:
# Rating Distribution
rating_distribution = df['rating'].value_counts().reset_index().sort_values('count', ascending=True)
rating_distribution.columns = ['rating', 'count']

fig = go.Figure(data=[go.Bar(
    y=rating_distribution['rating'].str.upper(),
    x=rating_distribution['count'],
    orientation='h',
    marker=dict(color='#E50914')
)])

fig.update_layout(
    title='Netflix Content Distribution by Rating',
    xaxis_title='Number of Titles',
    yaxis_title='Rating',
    height=500
)

fig.show()

print("Rating Distribution:")
print(rating_distribution.sort_values('count', ascending=False))
print(f"\nTop 5 Ratings:")

Rating Distribution:
       rating  count
0       tv-ma   3183
1       tv-14   2133
2       tv-pg    838
3           r    799
4       pg-13    490
5       tv-y7    330
6        tv-y    300
7          pg    287
8        tv-g    212
9          nr     78
10          g     41
11   tv-y7-fv      5
12  not rated      4
13      nc-17      3
14         ur      3

Top 5 Ratings:


In [ ]:
# Genre Distribution (split listed_in which contains comma-separated genres)
genres_expanded = df['listed_in'].str.split(', ').explode().str.strip().reset_index(drop=True)
genre_distribution = genres_expanded.value_counts().head(15).reset_index()
genre_distribution.columns = ['genre', 'count']

fig = go.Figure(data=[go.Bar(
    y=genre_distribution['genre'],
    x=genre_distribution['count'],
    orientation='h',
    marker=dict(color='#221F1F')
)])

fig.update_layout(
    title='Top 15 Netflix Genres',
    xaxis_title='Number of Titles',
    yaxis_title='Genre',
    height=600
)

fig.show()

print("Top Genres Distribution:")
print(genre_distribution)

Top Genres Distribution:
                       genre  count
0       international movies   2752
1                     dramas   2427
2                   comedies   1674
3     international tv shows   1328
4              documentaries    869
5         action & adventure    859
6         independent movies    756
7                  tv dramas    739
8   children & family movies    641
9            romantic movies    616
10                 thrillers    577
11               tv comedies    550
12            crime tv shows    459
13                  kids' tv    433
14                docuseries    380


In [ ]:
# Relationship between Content Type and Rating
type_rating = df.groupby(['type', 'rating']).size().reset_index(name='count')
type_rating_pivot = type_rating.pivot(index='rating', columns='type', values='count').fillna(0)

fig = go.Figure()

for content_type in type_rating_pivot.columns:
    fig.add_trace(go.Bar(
        name=content_type.title(),
        x=type_rating_pivot.index,
        y=type_rating_pivot[content_type],
    ))

fig.update_layout(
    title='Netflix Content: Type vs Rating Distribution',
    xaxis_title='Rating',
    yaxis_title='Number of Titles',
    barmode='group',
    template='plotly_white',
    height=500
)

fig.show()


print("Content Type by Rating:")

Content Type by Rating:


In [ ]:
# Summary Statistics for Task 2
print("=" * 60)
print("DISTRIBUTION SUMMARY")
print("=" * 60)
print(f"\nContent Type Breakdown:")
print(f"  • Movies: {(type_distribution[type_distribution['type']=='movie']['count'].values[0] / len(df) * 100):.1f}%")
print(f"  • TV Shows: {(type_distribution[type_distribution['type']=='tv show']['count'].values[0] / len(df) * 100):.1f}%")

print(f"\nRating Distribution:")
top_ratings = df['rating'].value_counts().head(3)
for rating, count in top_ratings.items():
    print(f"  • {rating.upper()}: {count} titles ({count/len(df)*100:.1f}%)")

print(f"\nGenre Insights:")
print(f"  • Total unique genres: {genres_expanded.nunique()}")
print(f"  • Most common genre: {genre_distribution.iloc[0]['genre']} ({genre_distribution.iloc[0]['count']} titles)")
print(f"  • Top 3 genres account for: {genre_distribution.head(3)['count'].sum() / len(df) * 100:.1f}% of content")

DISTRIBUTION SUMMARY

Content Type Breakdown:
  • Movies: 70.4%
  • TV Shows: 29.6%

Rating Distribution:
  • TV-MA: 3183 titles (36.6%)
  • TV-14: 2133 titles (24.5%)
  • TV-PG: 838 titles (9.6%)

Genre Insights:
  • Total unique genres: 42
  • Most common genre: international movies (2752 titles)
  • Top 3 genres account for: 78.7% of content


# Task 3: Identify Country-Level Content Contributions

In [ ]:
# Process country data (split comma-separated values)
countries_expanded = df['country'].str.split(', ').explode().str.strip().reset_index(drop=True)
country_distribution = countries_expanded.value_counts().reset_index()
country_distribution.columns = ['country', 'count']

print(f"Total unique countries: {countries_expanded.nunique()}")
print(f"\nTop 20 Countries by Content Contribution:")
print(country_distribution.head(20))

Total unique countries: 128

Top 20 Countries by Content Contribution:
           country  count
0    united states   3639
1            india   1045
2    not available    827
3   united kingdom    785
4           canada    432
5           france    389
6            japan    314
7            spain    228
8      south korea    226
9          germany    225
10          mexico    169
11           china    162
12       australia    157
13           egypt    117
14          turkey    113
15       hong kong    105
16         nigeria    103
17           italy     98
18          brazil     97
19       argentina     91


In [ ]:
# Visualize Top 20 Countries
top_countries = country_distribution.head(20).sort_values('count', ascending=True)

fig = go.Figure(data=[go.Bar(
    y=top_countries['country'],
    x=top_countries['count'],
    orientation='h',
    marker=dict(color='#E50914')
)])

fig.update_layout(
    title='Top 20 Countries Contributing Netflix Content',
    xaxis_title='Number of Titles',
    yaxis_title='Country',
    height=600
)

fig.show()

In [ ]:
# Analyze top countries by content type
df_countries = df.copy()
df_countries['country_list'] = df_countries['country'].str.split(', ')
df_countries = df_countries.explode('country_list')
df_countries['country_list'] = df_countries['country_list'].str.strip()

top_10_countries = country_distribution.head(10)['country'].tolist()
top_countries_data = df_countries[df_countries['country_list'].isin(top_10_countries)]

country_type = top_countries_data.groupby(['country_list', 'type']).size().reset_index(name='count')
country_type_pivot = country_type.pivot(index='country_list', columns='type', values='count').fillna(0)

# Sort by total content
country_type_pivot['Total'] = country_type_pivot.sum(axis=1)
country_type_pivot = country_type_pivot.sort_values('Total', ascending=True)

fig = go.Figure()

for content_type in ['movie', 'tv show']:
    if content_type in country_type_pivot.columns:
        fig.add_trace(go.Bar(
            name=content_type.title(),
            y=country_type_pivot.index,
            x=country_type_pivot[content_type],
            orientation='h'
        ))

fig.update_layout(
    title='Top 10 Countries: Movie vs TV Show Contributions',
    xaxis_title='Number of Titles',
    yaxis_title='Country',
    barmode='stack',
    height=500
)

fig.show()

In [ ]:
# Geographic concentration analysis
top_5_countries = country_distribution.head(5)
top_5_share = top_5_countries['count'].sum() / countries_expanded.count() * 100

top_10_countries_dist = country_distribution.head(10)
top_10_share = top_10_countries_dist['count'].sum() / countries_expanded.count() * 100

fig = go.Figure()

fig.add_trace(go.Pie(
    labels=['Top 5 Countries', 'Top 6-10 Countries', 'Other Countries'],
    values=[
        top_5_countries['count'].sum(),
        top_10_countries_dist.iloc[5:]['count'].sum(),
        country_distribution.iloc[10:]['count'].sum()
    ],
    marker=dict(colors=['#E50914', '#221F1F', '#B81D13'])
))

fig.update_layout(
    title='Market Concentration: Top Countries Share',
    height=500
)

fig.show()

print("Market Concentration Analysis:")
print(f"Top 5 countries account for: {top_5_share:.1f}% of all content")
print(f"Top 10 countries account for: {top_10_share:.1f}% of all content")
print(f"\nMarket Share of Top 5:")
for idx, row in top_5_countries.iterrows():
    share = row['count'] / countries_expanded.count() * 100
    print(f"  {row['country']}: {row['count']} titles ({share:.1f}%)")

Market Concentration Analysis:
Top 5 countries account for: 62.7% of all content
Top 10 countries account for: 75.6% of all content

Market Share of Top 5:
  united states: 3639 titles (33.9%)
  india: 1045 titles (9.7%)
  not available: 827 titles (7.7%)
  united kingdom: 785 titles (7.3%)
  canada: 432 titles (4.0%)


In [ ]:
# Growth trends by top countries over time
top_5_countries_list = country_distribution.head(5)['country'].tolist()
top_countries_growth = df_countries[df_countries['country_list'].isin(top_5_countries_list)].copy()

country_year_growth = top_countries_growth.groupby(['country_list', 'year_added']).size().reset_index(name='count')

fig = go.Figure()

for country in top_5_countries_list:
    country_data = country_year_growth[country_year_growth['country_list'] == country]
    fig.add_trace(go.Scatter(
        x=country_data['year_added'],
        y=country_data['count'],
        mode='lines+markers',
        name=country,
        line=dict(width=2),
        marker=dict(size=6)
    ))

fig.update_layout(
    title='Content Growth Trend: Top 5 Countries Over Time',
    xaxis_title='Year Added',
    yaxis_title='Number of Titles Added',
    hovermode='x unified',
    template='plotly_white',
    height=500
)

fig.show()

print("Top 5 Countries Content Growth Summary:")
for country in top_5_countries_list:
    country_total = top_countries_growth[top_countries_growth['country_list'] == country].shape[0]

Top 5 Countries Content Growth Summary:


In [ ]:
# Summary Statistics for Task 3
print("=" * 70)
print("COUNTRY-LEVEL CONTENT CONTRIBUTIONS SUMMARY")
print("=" * 70)
print(f"\nTotal unique countries: {countries_expanded.nunique()}")
print(f"Total country-title associations: {countries_expanded.count()}")
print(f"\nTop Country: {country_distribution.iloc[0]['country']} with {country_distribution.iloc[0]['count']} titles")

top_5 = country_distribution.head(5)
concentration = (top_5['count'].sum() / countries_expanded.count() * 100)
print(f"\nTop 5 countries account for {concentration:.1f}% of all content contributions")

print("\nTop 10 Countries Ranking:")
for idx, row in country_distribution.head(10).iterrows():
    percentage = (row['count'] / countries_expanded.count() * 100)
    print(f"  {idx+1:2d}. {row['country']:20s} - {row['count']:4d} titles ({percentage:5.1f}%)")

COUNTRY-LEVEL CONTENT CONTRIBUTIONS SUMMARY

Total unique countries: 128
Total country-title associations: 10731

Top Country: united states with 3639 titles

Top 5 countries account for 62.7% of all content contributions

Top 10 Countries Ranking:
   1. united states        - 3639 titles ( 33.9%)
   2. india                - 1045 titles (  9.7%)
   3. not available        -  827 titles (  7.7%)
   4. united kingdom       -  785 titles (  7.3%)
   5. canada               -  432 titles (  4.0%)
   6. france               -  389 titles (  3.6%)
   7. japan                -  314 titles (  2.9%)
   8. spain                -  228 titles (  2.1%)
   9. south korea          -  226 titles (  2.1%)
  10. germany              -  225 titles (  2.1%)


# Task 4: Create Derived Features (Content Length Category & Original vs. Licensed)

In [ ]:
# Feature 1: Content Length Category
# Categorize movies by duration and TV shows by number of seasons

df['content_length_category'] = 'Unknown'

# Movies: categorize by duration in minutes
movie_mask = df['type'] == 'movie'
df.loc[movie_mask & (df['duration_value'] < 90), 'content_length_category'] = 'Short (< 90 min)'
df.loc[movie_mask & (df['duration_value'] >= 90) & (df['duration_value'] < 120), 'content_length_category'] = 'Standard (90-120 min)'
df.loc[movie_mask & (df['duration_value'] >= 120), 'content_length_category'] = 'Long (> 120 min)'

# TV Shows: categorize by number of seasons
tv_mask = df['type'] == 'tv show'
df.loc[tv_mask & (df['duration_value'] == 1), 'content_length_category'] = 'Single Season'
df.loc[tv_mask & (df['duration_value'] >= 2) & (df['duration_value'] <= 5), 'content_length_category'] = 'Few Seasons (2-5)'
df.loc[tv_mask & (df['duration_value'] > 5), 'content_length_category'] = 'Many Seasons (> 5)'

print("Content Length Categories Created:")
print(df['content_length_category'].value_counts())
print("\nSample data:")
print(df[['type', 'duration_value', 'duration_unit', 'content_length_category']].head(10))

Content Length Categories Created:
content_length_category
Standard (90-120 min)    3092
Short (< 90 min)         1838
Single Season            1791
Long (> 120 min)         1198
Few Seasons (2-5)         706
Many Seasons (> 5)         81
Name: count, dtype: int64

Sample data:
      type  duration_value duration_unit content_length_category
0    movie              90           min   Standard (90-120 min)
1  tv show               2       Seasons       Few Seasons (2-5)
2  tv show               1        Season           Single Season
3  tv show               1        Season           Single Season
4  tv show               2       Seasons       Few Seasons (2-5)
5  tv show               1        Season           Single Season
6    movie              91           min   Standard (90-120 min)
7    movie             125           min        Long (> 120 min)
8  tv show               9       Seasons      Many Seasons (> 5)
9    movie             104           min   Standard (90-120 min)


In [ ]:
# Feature 2: Original vs. Licensed Content
# Infer "Original" content: if added within 1 year of release (Netflix originals)
# Legacy content: if there's a 2+ year gap between release and addition

df['year_released'] = df['release_year'].dt.year
df['year_added_col'] = df['date_added'].dt.year

df['content_origin'] = 'Licensed'

# Original content: Released and added in same year or added within 1 year of release
original_mask = (df['year_added_col'] - df['year_released']).abs() <= 1
df.loc[original_mask, 'content_origin'] = 'Original/Recent'

# Very old content: Released more than 5 years before being added
old_mask = (df['year_added_col'] - df['year_released']) >= 5
df.loc[old_mask, 'content_origin'] = 'Licensed Legacy'

# Content released in same year as added (likely Netflix Originals)
netflix_original_mask = df['year_added_col'] == df['year_released']
df.loc[netflix_original_mask, 'content_origin'] = 'Netflix Original'

print("Content Origin Distribution:")
print(df['content_origin'].value_counts())
print("\nPercentage:")
print(df['content_origin'].value_counts(normalize=True) * 100)

print("\nSample data:")
print(df[['title', 'year_released', 'year_added_col', 'content_origin']].head(10))

Content Origin Distribution:
content_origin
Netflix Original    3221
Licensed Legacy     2364
Original/Recent     1570
Licensed            1551
Name: count, dtype: int64

Percentage:
content_origin
Netflix Original    36.997473
Licensed Legacy     27.153687
Original/Recent     18.033540
Licensed            17.815300
Name: proportion, dtype: float64

Sample data:
                              title  year_released  year_added_col  \
0              Dick Johnson Is Dead           2020            2021   
1                     Blood & Water           2021            2021   
2                         Ganglands           2021            2021   
3             Jailbirds New Orleans           2021            2021   
4                      Kota Factory           2021            2021   
5                     Midnight Mass           2021            2021   
6  My Little Pony: A New Generation           2021            2021   
7                           Sankofa           1993            2021   
8    

In [ ]:
# Visualization 1: Content Length Category Distribution
length_category_dist = df['content_length_category'].value_counts().reset_index()
length_category_dist.columns = ['category', 'count']

fig = go.Figure(data=[go.Pie(
    labels=length_category_dist['category'],
    values=length_category_dist['count'],
    marker=dict(colors=['#E50914', '#221F1F', '#B81D13', '#564D4D', '#831F1C', '#F5B041'])
)])

fig.update_layout(
    title='Netflix Content Distribution by Length Category',
    height=500
)

fig.show()
fig.write_image("graph.png")

print("Content Length Category Summary:")
print(length_category_dist)
print(f"\nPercentage:")
for idx, row in length_category_dist.iterrows():
    pct = row['count'] / length_category_dist['count'].sum() * 100

Content Length Category Summary:
                category  count
0  Standard (90-120 min)   3092
1       Short (< 90 min)   1838
2          Single Season   1791
3       Long (> 120 min)   1198
4      Few Seasons (2-5)    706
5     Many Seasons (> 5)     81

Percentage:


In [ ]:
# Visualization 2: Original vs. Licensed Content
origin_dist = df['content_origin'].value_counts().reset_index()
origin_dist.columns = ['origin', 'count']

fig = go.Figure(data=[go.Bar(
    x=origin_dist['origin'],
    y=origin_dist['count'],
    marker=dict(color=['#E50914', '#221F1F', '#B81D13', '#564D4D'])
)])

fig.update_layout(
    title='Netflix Content: Original vs. Licensed Breakdown',
    xaxis_title='Content Origin',
    yaxis_title='Number of Titles',
    template='plotly_white',
    height=500
)

fig.show()

print("Content Origin Summary:")
print(origin_dist)
print(f"\nPercentage:")
for idx, row in origin_dist.iterrows():
    pct = row['count'] / origin_dist['count'].sum() * 100

Content Origin Summary:
             origin  count
0  Netflix Original   3221
1   Licensed Legacy   2364
2   Original/Recent   1570
3          Licensed   1551

Percentage:


In [ ]:
# Visualization 3: Content Type vs. Length Category
type_length = df.groupby(['type', 'content_length_category']).size().reset_index(name='count')

fig = go.Figure()

for content_type in df['type'].unique():
    data = type_length[type_length['type'] == content_type].sort_values('count', ascending=True)
    fig.add_trace(go.Bar(
        name=content_type.title(),
        y=data['content_length_category'],
        x=data['count'],
        orientation='h'
    ))

fig.update_layout(
    title='Content Type Distribution by Length Category',
    xaxis_title='Number of Titles',
    yaxis_title='Length Category',
    barmode='group',
    template='plotly_white',
    height=550
)

fig.show()

print("Content Type by Length Category:")

Content Type by Length Category:


In [ ]:
# Visualization 4: Content Origin vs. Type
origin_type = df.groupby(['content_origin', 'type']).size().reset_index(name='count')

fig = go.Figure()

for content_type in df['type'].unique():
    data = origin_type[origin_type['type'] == content_type]
    fig.add_trace(go.Bar(
        name=content_type.title(),
        x=data['content_origin'],
        y=data['count']
    ))

fig.update_layout(
    title='Netflix Content: Origin by Type (Movies vs TV Shows)',
    xaxis_title='Content Origin',
    yaxis_title='Number of Titles',
    barmode='group',
    template='plotly_white',
    height=500
)

fig.show()

print("Content Origin by Type:")

Content Origin by Type:


In [ ]:
# Summary Statistics for Task 4 - Derived Features
print("=" * 70)
print("DERIVED FEATURES SUMMARY")
print("=" * 70)

print("\n1. CONTENT LENGTH CATEGORY INSIGHTS:")
print("-" * 70)
movies = df[df['type'] == 'movie']
tv_shows = df[df['type'] == 'tv show']

print(f"\nMovies ({len(movies)} titles):")
movie_lengths = movies['content_length_category'].value_counts()
for cat, count in movie_lengths.items():
    pct = count / len(movies) * 100
    print(f"  • {cat}: {count} movies ({pct:.1f}%)")

print(f"\nTV Shows ({len(tv_shows)} titles):")
tv_lengths = tv_shows['content_length_category'].value_counts()
for cat, count in tv_lengths.items():
    pct = count / len(tv_shows) * 100
    print(f"  • {cat}: {count} shows ({pct:.1f}%)")

print("\n2. ORIGINAL VS. LICENSED CONTENT INSIGHTS:")
print("-" * 70)
origin_summary = df['content_origin'].value_counts()
for origin, count in origin_summary.items():
    pct = count / len(df) * 100
    print(f"  • {origin}: {count} titles ({pct:.1f}%)")

netflix_originals = df[df['content_origin'] == 'Netflix Original']
print(f"\nNetflix Originals are {len(netflix_originals) / len(df) * 100:.1f}% of content")

print("\n3. CONTENT TYPE DISTRIBUTION IN DERIVED FEATURES:")
print("-" * 70)
print(f"Movies with Standard Duration (90-120 min): {len(movies[movies['content_length_category'] == 'Standard (90-120 min)']) / len(movies) * 100:.1f}%")
print(f"TV Shows with Multiple Seasons (> 5): {len(tv_shows[tv_shows['content_length_category'] == 'Many Seasons (> 5)']) / len(tv_shows) * 100:.1f}%")
print(f"Original/Recent Movies: {len(movies[movies['content_origin'].isin(['Original/Recent', 'Netflix Original'])]) / len(movies) * 100:.1f}%")
print(f"Licensed TV Shows: {len(tv_shows[tv_shows['content_origin'] == 'Licensed']) / len(tv_shows) * 100:.1f}%")

# Show the new columns in the dataframe
print("\n4. NEW FEATURES ADDED TO DATASET:")
print("-" * 70)
print(f"✓ content_length_category: Categorizes movies by duration, TV shows by seasons")
print(f"✓ content_origin: Identifies Netflix Originals, Original/Recent, Licensed, or Licensed Legacy")
print(f"✓ year_released: Extracted from release_year for easier analysis")
print(f"✓ year_added_col: Extracted from date_added for easier analysis")

DERIVED FEATURES SUMMARY

1. CONTENT LENGTH CATEGORY INSIGHTS:
----------------------------------------------------------------------

Movies (6128 titles):
  • Standard (90-120 min): 3092 movies (50.5%)
  • Short (< 90 min): 1838 movies (30.0%)
  • Long (> 120 min): 1198 movies (19.5%)

TV Shows (2578 titles):
  • Single Season: 1791 shows (69.5%)
  • Few Seasons (2-5): 706 shows (27.4%)
  • Many Seasons (> 5): 81 shows (3.1%)

2. ORIGINAL VS. LICENSED CONTENT INSIGHTS:
----------------------------------------------------------------------
  • Netflix Original: 3221 titles (37.0%)
  • Licensed Legacy: 2364 titles (27.2%)
  • Original/Recent: 1570 titles (18.0%)
  • Licensed: 1551 titles (17.8%)

Netflix Originals are 37.0% of content

3. CONTENT TYPE DISTRIBUTION IN DERIVED FEATURES:
----------------------------------------------------------------------
Movies with Standard Duration (90-120 min): 50.5%
TV Shows with Multiple Seasons (> 5): 3.1%
Original/Recent Movies: 49.6%
Licensed T

In [ ]:
# Final Dataset Preview with All Features
print("\nFinal Dataset with All Engineered Features:")
print("=" * 70)
display(df[['title', 'type', 'rating', 'duration_value', 'duration_unit', 
            'content_length_category', 'content_origin', 'year_released', 'year_added_col']].head(15))

print(f"\nFinal Dataset Shape: {df.shape}")
print(f"Total Features: {len(df.columns)}")
print("\nAll columns in the dataset:")
for i, col in enumerate(df.columns, 1):
    print(f"  {i:2d}. {col}")


Final Dataset with All Engineered Features:


,title,type,rating,duration_value,duration_unit,content_length_category,content_origin,year_released,year_added_col
0,Dick Johnson Is Dead,movie,pg-13,90,min,Standard (90-120 min),Original/Recent,2020,2021
1,Blood & Water,tv show,tv-ma,2,Seasons,Few Seasons (2-5),Netflix Original,2021,2021
2,Ganglands,tv show,tv-ma,1,Season,Single Season,Netflix Original,2021,2021
3,Jailbirds New Orleans,tv show,tv-ma,1,Season,Single Season,Netflix Original,2021,2021
4,Kota Factory,tv show,tv-ma,2,Seasons,Few Seasons (2-5),Netflix Original,2021,2021
5,Midnight Mass,tv show,tv-ma,1,Season,Single Season,Netflix Original,2021,2021
6,My Little Pony: A New Generation,movie,pg,91,min,Standard (90-120 min),Netflix Original,2021,2021
7,Sankofa,movie,tv-ma,125,min,Long (> 120 min),Licensed Legacy,1993,2021
8,The Great British Baking Show,tv show,tv-14,9,Seasons,Many Seasons (> 5),Netflix Original,2021,2021
9,The Starling,movie,pg-13,104,min,Standard (90-120 min),Netflix Original,2021,2021



Final Dataset Shape: (8706, 19)
Total Features: 19

All columns in the dataset:
   1. show_id
   2. type
   3. title
   4. director
   5. cast
   6. country
   7. date_added
   8. release_year
   9. rating
  10. duration
  11. listed_in
  12. description
  13. duration_value
  14. duration_unit
  15. year_added
  16. content_length_category
  17. year_released
  18. year_added_col
  19. content_origin
